# 🚀 Notebook 03 — Vertex AI Deployment
**FMCG Promotional Analytics | Portfolio Project**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Panosemmanouilidis2/fmcg-ds-technical-portfolio/blob/main/promotional-analytics/notebooks/03_model_training.ipynb)

Deploys the trained XGBoost model to a live Google Cloud Vertex AI endpoint.

## ✅ Before You Start — Checklist

| Step | Action |
|---|---|
| 1 | Run `02_feature_engineering.ipynb` — confirms `models/xgb_tuned.json` exists |
| 2 | Create a GCP project at [console.cloud.google.com](https://console.cloud.google.com) |
| 3 | Enable **Vertex AI API** and **Cloud Storage API** in your project |
| 4 | Run `gcloud auth application-default login` in your terminal |
| 5 | Fill in **Cell 0** below — only block you need to edit |

**Run time after setup:** ~15 minutes


---

> ⚠️ **Cloud setup required for this notebook**
> This notebook deploys the model to Google Cloud Vertex AI.
> It requires a GCP account, Vertex AI API enabled, and credentials configured.
> **If you do not have a GCP account, skip to ** —
> notebooks 01, 02 and 04 run entirely in Colab with no cloud setup needed.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 0 — YOUR GCP CONFIGURATION  ← only block you need to edit
# ══════════════════════════════════════════════════════════════════

PROJECT_ID  = "YOUR_PROJECT_ID"     # GCP Console → project name at the top
REGION      = "europe-west2"        # change only if your project is in a different region
BUCKET_NAME = "YOUR_BUCKET_NAME"    # new or existing GCS bucket (no gs:// prefix)
MODEL_NAME  = "fmcg-xgb-promo-v1"  # display name in Vertex AI Model Registry

# ── Derived — do not edit below this line ─────────────────────────
BUCKET_URI        = f"gs://{BUCKET_NAME}"
GCS_MODEL_DIR     = f"{BUCKET_URI}/artifacts/model"
SKLEARN_CONTAINER = "us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-0:latest"

print("=== CONFIG ===")
print(f"  Project  : {PROJECT_ID}")
print(f"  Region   : {REGION}")
print(f"  Bucket   : {BUCKET_URI}")
print(f"  Model    : {MODEL_NAME}")
if "YOUR_" in PROJECT_ID or "YOUR_" in BUCKET_NAME:
    print("\n⚠️  Fill in PROJECT_ID and BUCKET_NAME above before continuing")
else:
    print("\n✅ Config looks good — proceed to Cell 1")


## Step 1 — Install & Authenticate

In [ ]:
!pip install google-cloud-aiplatform google-cloud-storage xgboost==1.7.6 --quiet
print("✅ Libraries installed")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 2 — AUTHENTICATE
# On Vertex AI Workbench: skip — credentials are automatic.
# On Colab or local: run  gcloud auth application-default login
# in your terminal first, then run this cell.
# ══════════════════════════════════════════════════════════════════
import google.auth, google.auth.transport.requests

credentials, detected_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
credentials.refresh(google.auth.transport.requests.Request())
print(f"✅ Authenticated — detected project: {detected_project}")


In [ ]:
import os, json, pickle, shutil
import numpy as np
import xgboost as xgb
from google.cloud import aiplatform, storage

aiplatform.init(project=PROJECT_ID, location=REGION)
print(f"✅ Vertex AI SDK initialised — {PROJECT_ID} / {REGION}")


## Step 2 — Upload Model Artefacts to GCS

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 4 — CREATE BUCKET  (skips if already exists)
# ══════════════════════════════════════════════════════════════════
client = storage.Client(project=PROJECT_ID)
try:
    bucket = client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"✅ Bucket created: {BUCKET_URI}")
except Exception as e:
    if "already own" in str(e).lower() or "already exists" in str(e).lower():
        bucket = client.bucket(BUCKET_NAME)
        print(f"✅ Bucket already exists: {BUCKET_URI}")
    else:
        raise


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 5 — STAGE & UPLOAD MODEL
# ══════════════════════════════════════════════════════════════════
os.makedirs('staging/model', exist_ok=True)

booster = xgb.Booster()
booster.load_model('models/xgb_tuned.json')
booster.save_model('staging/model/model.json')

for f in ['feature_cols.json', 'feature_medians.json']:
    shutil.copy(f'models/{f}', f'staging/model/{f}')

for fname in os.listdir('staging/model'):
    blob_name = f'artifacts/model/{fname}'
    bucket.blob(blob_name).upload_from_filename(f'staging/model/{fname}')
    print(f"  Uploaded → gs://{BUCKET_NAME}/{blob_name}")

print(f"\n✅ All artefacts uploaded to {GCS_MODEL_DIR}")


## Step 3 — Register, Deploy & Test

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 6 — REGISTER MODEL  (⏳ 3–5 minutes)
# ══════════════════════════════════════════════════════════════════
print(f"Registering '{MODEL_NAME}'...")

model = aiplatform.Model.upload(
    display_name                = MODEL_NAME,
    artifact_uri                = GCS_MODEL_DIR,
    serving_container_image_uri = SKLEARN_CONTAINER,
    description                 = "XGBoost promotional sales forecaster — Market A & B",
)
print(f"✅ Model registered: {model.resource_name}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 7 — CREATE ENDPOINT  (⏳ 2–3 minutes)
# ══════════════════════════════════════════════════════════════════
endpoint = aiplatform.Endpoint.create(
    display_name = f"{MODEL_NAME}-endpoint",
    description  = "FMCG promotional sales forecaster"
)
with open('models/endpoint_resource_name.txt', 'w') as f:
    f.write(endpoint.resource_name)
print(f"✅ Endpoint created: {endpoint.resource_name}")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 8 — DEPLOY MODEL  (⏳ 10–15 minutes — do NOT close)
# ══════════════════════════════════════════════════════════════════
print("Deploying model — please wait 10–15 minutes...\n")

endpoint.deploy(
    model                       = model,
    deployed_model_display_name = f"{MODEL_NAME}-deployed",
    machine_type                = "n1-standard-2",
    min_replica_count           = 1,
    max_replica_count           = 1,
    traffic_percentage          = 100,
)
print("✅ Model deployed — endpoint is ACTIVE")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 9 — SMOKE TEST
# ══════════════════════════════════════════════════════════════════
import json, numpy as np

feature_cols = json.load(open('models/feature_cols.json'))
medians      = json.load(open('models/feature_medians.json'))
payload      = [float(medians.get(col, 0.0)) for col in feature_cols]

response   = endpoint.predict(instances=[payload])
pred_units = max(0, round(np.expm1(response.predictions[0])))

print("=== SMOKE TEST ===")
print(f"  Predicted sales volume : {pred_units:,} units")
print("✅ Endpoint live" if pred_units > 0 else "⚠️  Check feature alignment")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELL 10 — UNDEPLOY  (run at end of session to stop billing)
# Uncomment the lines below when ready to stop the endpoint.
# To redeploy later: re-run Cells 7–9 only (~15 min).
# ══════════════════════════════════════════════════════════════════

# for deployed_model in endpoint.list_models():
#     print(f"Undeploying: {deployed_model.display_name}...")
#     endpoint.undeploy(deployed_model_id=deployed_model.id)
# print("Endpoint stopped — billing halted")

print("ℹ️  Uncomment lines above to undeploy and stop billing")
